# Validation: quantifying error in detection, calibration, and clustering

*Using Projective Transformation for the Spatial Analysis of Team Behaviors in Football*

The other notebooks (`00`-`03`) take the pipeline's output at face value. This one instead
measures how much to trust it, for the thesis's validation/methodology section.

**Every section below runs automatically, top to bottom, with no manual input required** - same
as notebooks `00`-`03`. Each section has two layers:

1. **Automatic checks (always on).** No ground truth exists for this project's own footage, so
   these can't report "true" accuracy - but they need zero manual work and still catch real
   problems: is the pitch calibration's error small when tested on points it wasn't fit from,
   are per-class detection counts/confidences stable across the clip, are the two team colour
   clusters actually well-separated.
2. **Optional manual sections (off by default, like `USE_MANUAL_FALLBACK` in notebook `00`).**
   Flip a `RUN_MANUAL_...` flag to `True` only if you want a real, precise number (precision/recall
   against hand-labeled boxes, clustering accuracy against hand-labeled true team) for the thesis -
   this needs a few minutes of hand-labeling. Leave every flag `False` and the notebook is fully
   automatic end to end.

## 1. Clone the repository and install dependencies

In [ ]:
import os
import sys

REPO_URL = "https://github.com/Batomet/Magisterka.git"
BRANCH = "claude/field-position-detection-dqda1g"
REPO_DIR = "/content/Magisterka"

if not os.path.isdir(REPO_DIR):
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git pull
!git log -1 --oneline

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR + "/src")

In [ ]:
!pip install -q -r requirements.txt

## 2. Mount Google Drive and pick a clip

In [ ]:
from pitchvision import DriveConfig, mount_drive

mount_drive()

# Adjust `root` if BuildingAction/Goals/SetPieces don't live directly under My Drive.
drive_cfg = DriveConfig(root="/content/drive/MyDrive/Magisterka")
print(drive_cfg.building_action_path)
print(drive_cfg.goals_path)
print(drive_cfg.set_pieces_path)

In [ ]:
from pitchvision import VideoFrames, list_videos

goal_videos = list_videos(drive_cfg.goals_path)
print(f"Found {len(goal_videos)} videos in Goals/")

sample_video = goal_videos[0]
frames = VideoFrames(sample_video)
print(sample_video, "-", frames.frame_count, "frames @", frames.fps, "fps")

## 3. Pitch calibration accuracy

Fully automatic - no manual point-picking. `PitchKeypointDetector` already finds pitch landmarks
in the frame automatically, and each one's TRUE pitch position (in metres) is not something a
human needs to label - it's a known constant from the Laws of the Game (`pitch_keypoint_template()`).
So instead of hand-clicking landmarks, `compute_calibration_holdout_error` holds out a random
subset of the *automatically detected* confident keypoints and measures reprojection error
(metres) on them - the same held-out-error idea as before, just with zero typing.

(`PitchCalibrator.reprojection_error` alone would measure error on the exact points the homography
was fit from, which always looks good and says nothing about accuracy elsewhere on the pitch -
that's why this holds points out rather than reusing the fit points.)

In [ ]:
from pitchvision import (
    PitchKeypointDetector,
    compute_calibration_holdout_error,
    download_pitch_keypoint_weights,
    pitch_keypoint_template,
)

weights_path = download_pitch_keypoint_weights(
    "/content/drive/MyDrive/pitchvision_models/football-pitch-detection.pt"
)
keypoint_detector = PitchKeypointDetector(weights=weights_path, confidence=0.5)

first_frame = frames.read_frame(0)
xy, conf = keypoint_detector.detect(first_frame)
template = pitch_keypoint_template()
confident = conf >= keypoint_detector.confidence
print(f"{confident.sum()}/32 pitch keypoints detected automatically with confidence >= {keypoint_detector.confidence}")

MIN_POINTS_FOR_HOLDOUT = 8  # enough margin to both fit (>=4) and hold out (>=1) meaningfully

if confident.sum() >= MIN_POINTS_FOR_HOLDOUT:
    holdout_result = compute_calibration_holdout_error(xy[confident], template[confident], n_repeats=30)
    print(f"\n{holdout_result['n_points']} auto-detected points, fit on {holdout_result['n_fit']} each of "
          f"{holdout_result['n_repeats']} random splits, held out on the rest:")
    print(f"  mean holdout error: {holdout_result['mean_error_m']:.3f} m")
    print(f"  std  holdout error: {holdout_result['std_error_m']:.3f} m")
    print(f"  max  holdout error: {holdout_result['max_error_m']:.3f} m")
else:
    holdout_result = None
    print("\nNot enough confident automatic keypoints for a reliable holdout split on this frame - "
          "see the optional manual fallback below.")

# The calibrator actually used for tracking in section 5 - fit on ALL confidently-detected points
# (the holdout evaluation above only ever fits on a random subset, to fairly measure accuracy on
# points it hasn't seen).
calibrator = keypoint_detector.calibrate(first_frame)

In [ ]:
import matplotlib.pyplot as plt

if holdout_result is not None:
    plt.figure(figsize=(6, 4))
    plt.hist(holdout_result["holdout_errors_m"], bins=15)
    plt.xlabel("Held-out reprojection error (m)")
    plt.ylabel("Count")
    plt.title("Distribution of held-out calibration error across random splits")
    plt.show()
else:
    print("Skipped - no holdout_result (see the message above).")

### Optional: manual calibration fallback

Off by default - only turn this on if section 3's automatic detection above couldn't find enough
confident keypoints on this clip (heavy occlusion, an unusual crop). Same hover-to-read-pixel-coordinates
pattern as notebook `00`'s manual fallback: hand-pick 10+ landmarks (more than the minimum 4 a fit
needs) so a holdout split is still possible, fill them into `landmark_pixels`, and this recomputes
`calibrator`/`holdout_result` from them instead.

In [ ]:
import cv2
import numpy as np
import plotly.express as px

from pitchvision import PITCH_LANDMARKS_M, PitchCalibrator

# Set this to True only if section 3's automatic detection couldn't find enough confident
# keypoints. With this False (the default), this cell is a no-op - safe to leave in place when
# running the whole notebook with Run All.
USE_MANUAL_CALIBRATION_FALLBACK = False

if not USE_MANUAL_CALIBRATION_FALLBACK:
    print("Skipping manual calibration fallback (USE_MANUAL_CALIBRATION_FALLBACK is False). "
          "Using the automatic calibrator from section 3.")
else:
    fig = px.imshow(cv2.cvtColor(first_frame, cv2.COLOR_BGR2RGB))
    fig.update_layout(title="Hover to read pixel coordinates for the landmarks below", height=700)
    fig.show()
    print("Available landmark names:", list(PITCH_LANDMARKS_M.keys()))

    # EDIT THIS: at least 10 landmarks, read off the hover tooltip above. Do not leave these
    # defaults - they are placeholders and do not correspond to real points in your video.
    landmark_pixels = {
        "top_left_corner": (50, 60),
        "top_right_corner": (1200, 55),
        "bottom_left_corner": (10, 650),
        "bottom_right_corner": (1250, 640),
        "centre_spot": (630, 340),
        "centre_top": (630, 55),
        "centre_bottom": (630, 650),
        "left_penalty_top": (150, 200),
        "left_penalty_bottom": (150, 480),
        "left_penalty_spot": (220, 340),
        "left_six_yard_top": (80, 260),
        "left_six_yard_bottom": (80, 420),
    }

    _PLACEHOLDER = {
        "top_left_corner": (50, 60),
        "top_right_corner": (1200, 55),
        "bottom_left_corner": (10, 650),
        "bottom_right_corner": (1250, 640),
        "centre_spot": (630, 340),
        "centre_top": (630, 55),
        "centre_bottom": (630, 650),
        "left_penalty_top": (150, 200),
        "left_penalty_bottom": (150, 480),
        "left_penalty_spot": (220, 340),
        "left_six_yard_top": (80, 260),
        "left_six_yard_bottom": (80, 420),
    }
    assert landmark_pixels != _PLACEHOLDER, (
        "landmark_pixels still holds the placeholder values - edit them with real pixel "
        "coordinates read off the hover tooltip above before continuing."
    )

    pixel_points = np.array(list(landmark_pixels.values()))
    pitch_points = np.array([PITCH_LANDMARKS_M[name] for name in landmark_pixels])

    holdout_result = compute_calibration_holdout_error(pixel_points, pitch_points, n_repeats=30)
    print(f"mean holdout error: {holdout_result['mean_error_m']:.3f} m")

    calibrator = PitchCalibrator.from_point_pairs(pixel_points, pitch_points)

## 4. Detection accuracy (YOLO)

**Automatic (always on):** no ground truth exists to compute real precision/recall against, but
detection quality problems usually show up as *instability* - a player count that swings wildly
frame to frame, or confidences clustered suspiciously low - so this samples frames across the clip
and reports per-class count/confidence statistics with zero manual work.

**Optional manual (off by default):** for a real precision/recall/F1 number, hand-label boxes on a
few frames and compare against the detector's own predictions there.

In [ ]:
from pitchvision import PlayerBallDetector, SPORTS_DETECTION_CLASSES, download_player_detection_weights

player_weights_path = download_player_detection_weights(
    "/content/drive/MyDrive/pitchvision_models/football-player-detection.pt"
)
detector = PlayerBallDetector(
    weights=player_weights_path, confidence=0.3, classes=SPORTS_DETECTION_CLASSES, imgsz=1280
)
SPECIALIZED_CLASS_NAMES = ("player", "goalkeeper", "referee", "ball")

In [ ]:
import pandas as pd

# Automatic, no labeling: sample frames across the clip and check how STABLE per-class counts and
# confidences are. Wild swings (e.g. player count jumping between 14 and 22) usually mean missed
# detections from occlusion/motion blur rather than an actual change in how many players are visible.
STRIDE = 10
MAX_CHECK_FRAMES = 120
check_frame_indices = list(range(0, min(frames.frame_count, MAX_CHECK_FRAMES), STRIDE))

count_rows, conf_rows = [], []
for i in check_frame_indices:
    frame = frames.read_frame(i)
    dets = detector.detect(frame)
    counts = {cls: 0 for cls in SPECIALIZED_CLASS_NAMES}
    for d in dets:
        counts[d.class_name] = counts.get(d.class_name, 0) + 1
        conf_rows.append({"frame": i, "class_name": d.class_name, "confidence": d.confidence})
    counts["frame"] = i
    count_rows.append(counts)

counts_df = pd.DataFrame(count_rows).set_index("frame")
conf_df = pd.DataFrame(conf_rows)

print(f"Sampled {len(check_frame_indices)} frames (every {STRIDE}th, up to frame {MAX_CHECK_FRAMES}).\n")
print("Detections per class per checked frame - mean / std (lower std = more consistent):")
print(counts_df.agg(["mean", "std"]).T)
print("\nConfidence per class - mean / std / n:")
print(conf_df.groupby("class_name")["confidence"].agg(["mean", "std", "count"]))

In [ ]:
counts_df.plot(figsize=(8, 4), marker="o")
plt.xlabel("frame")
plt.ylabel("detections")
plt.title("Per-class detection count across sampled frames")
plt.show()

### Optional: manual detection labeling (real precision/recall/F1)

Off by default. Pick a few frames, hover to read pixel coordinates for the boxes actually visible,
and compare against the detector's predictions on those same frames with IoU-matched
precision/recall/F1/mean-IoU per class.

In [ ]:
# Set this to True to hand-label a few frames for a real precision/recall number. With this False
# (the default), this section is a no-op - safe to leave in place when running with Run All.
RUN_MANUAL_DETECTION_LABELING = False

if RUN_MANUAL_DETECTION_LABELING:
    EVAL_FRAME_INDICES = sorted({0, frames.frame_count // 3, 2 * frames.frame_count // 3})
    eval_frames = {i: frames.read_frame(i) for i in EVAL_FRAME_INDICES}
    frame_detections = {i: detector.detect(frame) for i, frame in eval_frames.items()}
    for i, dets in frame_detections.items():
        print(f"frame {i}: {len(dets)} detections")
else:
    print("Skipping manual detection labeling (RUN_MANUAL_DETECTION_LABELING is False).")

In [ ]:
if RUN_MANUAL_DETECTION_LABELING:
    for i, frame in eval_frames.items():
        fig = px.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        fig.update_layout(title=f"Frame {i} - hover to read box corner pixel coordinates", height=700)
        fig.show()
else:
    print("Skipping (RUN_MANUAL_DETECTION_LABELING is False).")

In [ ]:
from pitchvision import compute_detection_metrics, ground_truth_boxes_dataframe, predicted_boxes_dataframe

if RUN_MANUAL_DETECTION_LABELING:
    # EDIT THIS: for each frame index above, list every (class_name, x1, y1, x2, y2) box you can
    # confidently place by eye. class_name must be one of "player", "goalkeeper", "referee", "ball".
    # Do not leave these defaults - they are placeholders and do not correspond to real boxes.
    ground_truth_boxes = {
        EVAL_FRAME_INDICES[0]: [
            ("player", 100, 200, 140, 320),
            ("player", 300, 180, 340, 300),
            ("ball", 500, 400, 515, 415),
        ],
    }

    _PLACEHOLDER = {
        EVAL_FRAME_INDICES[0]: [
            ("player", 100, 200, 140, 320),
            ("player", 300, 180, 340, 300),
            ("ball", 500, 400, 515, 415),
        ],
    }
    assert ground_truth_boxes != _PLACEHOLDER, (
        "ground_truth_boxes still holds the placeholder values - edit them with real boxes read "
        "off the hover tooltips above before continuing."
    )

    predicted_df = predicted_boxes_dataframe(frame_detections)
    ground_truth_df = ground_truth_boxes_dataframe(ground_truth_boxes)
    detection_metrics_df = compute_detection_metrics(predicted_df, ground_truth_df, iou_threshold=0.5)
else:
    detection_metrics_df = None
    print("Skipping (RUN_MANUAL_DETECTION_LABELING is False).")

detection_metrics_df

## 5. Team-colour clustering accuracy (K-means)

**Automatic (always on):** fits `TeamClassifier` as usual and reports the **silhouette score** - a
standard unsupervised cluster-quality metric (near `1` = the two colour clusters are well
separated, near `0`/negative = the clustering isn't finding real structure, e.g. lighting or grass
bleed-through dominating over actual kit colour) - plus cluster sizes, with zero manual work.

**Optional manual (off by default):** hand-label a sample of tracks' true team and compare against
the resolved `team_id` for a real accuracy number.

In [ ]:
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

from pitchvision import TeamClassifier, collect_jersey_samples

jersey_samples = collect_jersey_samples(sample_video, detector, class_names=("player",), stride=30)
jersey_colors = np.array([s.color for s in jersey_samples])
team_classifier = TeamClassifier(n_clusters=2).fit(jersey_colors)
print("cluster_swatches (RGB):", team_classifier.cluster_swatches)

cluster_labels = team_classifier.predict_from_colors(jersey_colors)
scaled = StandardScaler().fit_transform(jersey_colors)
sil = silhouette_score(scaled, cluster_labels)
sizes = np.bincount(cluster_labels)
print(f"\nsilhouette score: {sil:.3f} (near 1 = well-separated clusters, near 0/negative = the "
      "clustering isn't finding real structure)")
print(f"cluster sizes: {dict(enumerate(sizes))} (very lopsided, e.g. 90/10, is also a red flag)")

In [ ]:
from pitchvision import plot_jersey_color_samples

plot_jersey_color_samples(jersey_samples, cluster_labels, cluster_colors={0: "yellow", 1: "red"})

### Optional: manual clustering accuracy (real accuracy against hand-labeled tracks)

Off by default. Runs a short tracked window with the calibrator from section 3, shows a grid of
tracks' crops, and lets you hand-label true team per track - compared against the resolved
`team_id` under the optimal cluster-id-to-team matching.

In [ ]:
from pitchvision import PlayerTracker, TrackingPipeline

# Set this to True to hand-label a sample of tracks for a real clustering-accuracy number. With
# this False (the default), this section is a no-op.
RUN_MANUAL_CLUSTER_LABELING = False

if RUN_MANUAL_CLUSTER_LABELING:
    tracker = PlayerTracker(
        weights=player_weights_path, confidence=0.3, classes=SPORTS_DETECTION_CLASSES, imgsz=1280
    )
    pipeline = TrackingPipeline(
        tracker=tracker,
        calibrator=calibrator,
        team_classifier=team_classifier,
        team_eligible_class_names=("player",),
    )
    eval_tracks_df = pipeline.run(sample_video, max_frames=90)
    print(eval_tracks_df["class_name"].value_counts())
else:
    print("Skipping manual clustering labeling (RUN_MANUAL_CLUSTER_LABELING is False).")

In [ ]:
if RUN_MANUAL_CLUSTER_LABELING:
    N_TRACKS_TO_LABEL = 16

    player_rows = eval_tracks_df[eval_tracks_df["class_name"] == "player"]
    first_seen = player_rows.sort_values("frame").drop_duplicates("track_id")
    sample_tracks = first_seen.head(N_TRACKS_TO_LABEL)

    crops, track_ids = [], []
    for _, row in sample_tracks.iterrows():
        frame = frames.read_frame(int(row["frame"]))
        x1, y1, x2, y2 = int(row["bbox_x1"]), int(row["bbox_y1"]), int(row["bbox_x2"]), int(row["bbox_y2"])
        crops.append(frame[max(y1, 0):y2, max(x1, 0):x2])
        track_ids.append(int(row["track_id"]))

    n_cols = 4
    n_rows = (len(crops) + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(2.5 * n_cols, 2.5 * n_rows))
    axes = np.atleast_2d(axes)
    for idx in range(n_rows * n_cols):
        row, col = divmod(idx, n_cols)
        ax = axes[row, col]
        ax.set_xticks([])
        ax.set_yticks([])
        if idx >= len(crops):
            ax.axis("off")
            continue
        ax.imshow(cv2.cvtColor(crops[idx], cv2.COLOR_BGR2RGB))
        ax.set_title(f"track {track_ids[idx]}", fontsize=9)
    plt.tight_layout()
    plt.show()
    print("track_ids shown, in order:", track_ids)
else:
    print("Skipping (RUN_MANUAL_CLUSTER_LABELING is False).")

In [ ]:
from pitchvision import compute_clustering_accuracy

if RUN_MANUAL_CLUSTER_LABELING:
    # EDIT THIS: for as many of the track_ids printed above as you reasonably can (at least a
    # handful), the TRUE team by eye ("A" or "B" - which physical team, not a cluster id). Do not
    # leave these defaults - they are placeholders.
    true_team_by_track = {
        track_ids[0]: "A",
        track_ids[1]: "A",
        track_ids[2]: "B",
        track_ids[3]: "B",
    }

    _PLACEHOLDER = {
        track_ids[0]: "A",
        track_ids[1]: "A",
        track_ids[2]: "B",
        track_ids[3]: "B",
    }
    assert true_team_by_track != _PLACEHOLDER, (
        "true_team_by_track still holds the placeholder values - label the tracks shown above "
        "(or at least a handful of them) before continuing."
    )

    labeled_tracks = eval_tracks_df[eval_tracks_df["track_id"].isin(true_team_by_track)].drop_duplicates("track_id")
    true_labels = labeled_tracks["track_id"].map(true_team_by_track)
    predicted_labels = labeled_tracks["team_id"]

    clustering_result = compute_clustering_accuracy(true_labels, predicted_labels)
    print(f"accuracy: {clustering_result['accuracy']:.3f} ({clustering_result['n_samples']} labeled tracks)")
    print("cluster id -> true team:", clustering_result["cluster_to_true_label"])
else:
    clustering_result = None
    print("Skipping (RUN_MANUAL_CLUSTER_LABELING is False).")

clustering_result["confusion_matrix"] if clustering_result is not None else None

## Summary

Always-on, zero-labeling checks (report these even with every flag left `False`):

- **Calibration**: `holdout_result["mean_error_m"]` (± `std_error_m`) - held-out reprojection error
  in metres, from automatically-detected keypoints.
- **Detection**: `counts_df`/`conf_df` - per-class count and confidence stability across the clip.
- **Clustering**: `sil` (silhouette score) and cluster sizes.

If you switched on any `RUN_MANUAL_...`/`USE_MANUAL_...` flag for a precise number:

- **Detection**: `detection_metrics_df` - precision/recall/F1/mean IoU, per class and overall.
- **Clustering**: `clustering_result["accuracy"]` - team-assignment accuracy against hand-labeled tracks.

All of this came from ONE clip - re-run this notebook against a couple more clips (ideally one from
each of `BuildingAction`/`Goals`/`SetPieces`) before treating any of these numbers as representative
of the pipeline as a whole.